# Weather Dataset Overview

This notebook inspects the active 1,000-timestep datasets: 12 train, 8 finetune, and 8 test. Training contains static and slow-mixed weather only. The two random-mix tests change weather after reproducible random blocks of 100 to 200 timesteps.

In [ ]:
from collections import Counter
from pathlib import Path
import gzip
import json
import os
import xml.etree.ElementTree as ET

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-vec-cache")

import matplotlib.pyplot as plt
import numpy as np

In [ ]:
def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data" / "datasets").exists():
            return candidate
    raise FileNotFoundError("Could not find data/datasets.")


PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / "data" / "datasets"
CATEGORIES = ["train", "finetune", "test"]
WEATHERS = ["BASE", "RAIN", "SNOW", "FOG"]
WEATHER_CODE = {name: index for index, name in enumerate(WEATHERS)}
WEATHER_COLORS = ["#4f83cc", "#62a87c", "#d99a3d", "#777777"]

datasets = {
    category: sorted(path for path in (DATA_ROOT / category).iterdir() if path.is_dir())
    for category in CATEGORIES
}

for category in CATEGORIES:
    print(f"{category:8s}: {len(datasets[category])} datasets")

In [ ]:
def read_xml(path: Path):
    if path.suffix == ".gz":
        with gzip.open(path, "rb") as handle:
            return ET.fromstring(handle.read())
    return ET.parse(path).getroot()


def chunk_file(dataset: Path, kind: str) -> Path:
    matches = sorted((dataset / kind).glob("chunk_0.xml*"))
    if len(matches) != 1:
        raise FileNotFoundError(f"Expected one {kind} chunk in {dataset}")
    return matches[0]


def inspect_dataset(dataset: Path):
    metadata = json.loads((dataset / "metadata.json").read_text())
    vehicle_root = read_xml(chunk_file(dataset, "vehicles"))
    task_root = read_xml(chunk_file(dataset, "tasks"))

    labels = []
    for timestep in vehicle_root.findall("timestep"):
        vehicles = timestep.findall("vehicle")
        weather = Counter(
            vehicle.get("weather_scenario", "BASE") for vehicle in vehicles
        ).most_common(1)[0][0]
        labels.append(weather)

    task_counts = Counter()
    for timestep in task_root.findall("timestep"):
        for task in timestep.findall("task"):
            task_counts[task.get("weather_scenario", "BASE")] += 1

    return {
        "name": dataset.name,
        "stage": metadata["stage"],
        "duration": metadata["duration"],
        "order": [block["scenario"] for block in metadata["blocks"]],
        "labels": labels,
        "task_counts": task_counts,
    }


records = {
    category: [inspect_dataset(dataset) for dataset in datasets[category]]
    for category in CATEGORIES
}

In [ ]:
for category in CATEGORIES:
    print(f"\n{category.upper()}")
    for record in records[category]:
        order = " -> ".join(record["order"])
        total_tasks = sum(record["task_counts"].values())
        print(
            f"{record['name']:24s} stage={record['stage']:6s} "
            f"timesteps={record['duration']} tasks={total_tasks:6d} weather={order}"
        )

## Weather Timeline

In [ ]:
from matplotlib.colors import ListedColormap

weather_cmap = ListedColormap(WEATHER_COLORS)

for category in CATEGORIES:
    category_records = records[category]
    timeline = np.array([
        [WEATHER_CODE[label] for label in record["labels"]]
        for record in category_records
    ])
    fig, axis = plt.subplots(figsize=(13, max(3, len(category_records) * 0.45)))
    image = axis.imshow(timeline, aspect="auto", cmap=weather_cmap, vmin=0, vmax=3)
    axis.set_title(f"{category.upper()} weather timelines")
    axis.set_xlabel("simulation timestep")
    axis.set_yticks(range(len(category_records)))
    axis.set_yticklabels([record["name"] for record in category_records])
    colorbar = fig.colorbar(image, ax=axis, ticks=range(4), pad=0.02)
    colorbar.ax.set_yticklabels(WEATHERS)
    plt.tight_layout()
    plt.show()

## Generated Tasks By Weather

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(19, 5))

for axis, category in zip(axes, CATEGORIES):
    category_records = records[category]
    x = np.arange(len(category_records))
    bottom = np.zeros(len(category_records))
    for weather, color in zip(WEATHERS, WEATHER_COLORS):
        values = np.array([record["task_counts"].get(weather, 0) for record in category_records])
        axis.bar(x, values, bottom=bottom, label=weather, color=color)
        bottom += values
    axis.set_title(category.upper())
    axis.set_ylabel("task count")
    axis.set_xticks(x)
    axis.set_xticklabels([record["name"] for record in category_records], rotation=70, ha="right")

axes[-1].legend()
plt.tight_layout()
plt.show()